# Introducción

El objetivo de este cuaderno es el de desarrollar una aplicación que nos permita calcular y graficar diversis indicadores bursátiles, minando datos a partir de un servicio de API Rest de AlphaVantage.

Para ello estaremos utilizando librerías de cálculo de indicadores para enrriquecer el Dataset con nuevas características y librerías de graficación para mostrar en gráficas los resultados.

Por último, estaremos utilizando un modelo simple de ARIMA (modelo autoregresivo) para intentar predecir el precio de cierre a futuro de la acción elegida.

# **Paso 1.**

Instalamos todas las librerías y paquetes necesarios e importamos los métodos y librerías necesarias.

Antes de comenzar no olvides activar el entorno: .\\.venv\Scripts\Activate.ps1

Luego instala con _pip install ta plotly pmdarima_

In [3]:
import numpy as np
import pandas as pd

from datetime import datetime
from datetime import timedelta

import ta

from ta import add_all_ta_features
from ta.utils import dropna
from ta.trend import ADXIndicator
from ta.trend import MACD
from ta.trend import CCIIndicator
from ta.trend import EMAIndicator
from ta.trend import SMAIndicator
from ta.volume import OnBalanceVolumeIndicator
from ta.momentum import RSIIndicator
from ta.momentum import StochasticOscillator
from ta.momentum import StochRSIIndicator

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from pmdarima.arima import auto_arima

#**Paso 2.**

Cargamos los datos de IBM mediante la API de AlphaVantage.co.

En este caso estamos utilizando el símbolo por defecto (IBM). Si el usuario quiere analizar un símbolo diferente puede solicitar una API KEY gratuita [GET YOUR FREE API KEY TODAY](https://www.alphavantage.co/support/#api-key) para acceder a más de 200000 símbolos de distintas bolsas a nivel mundial.

In [4]:
url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo&datatype=csv"
df = pd.read_csv(url)
df.info()
df

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  100 non-null    str    
 1   open       100 non-null    float64
 2   high       100 non-null    float64
 3   low        100 non-null    float64
 4   close      100 non-null    float64
 5   volume     100 non-null    int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 4.8 KB


,timestamp,open,high,low,close,volume
0,2026-09-17,238.58,243.3650,237.3500,237.75,5905870
1,2026-09-16,240.50,243.3399,235.4100,237.49,6181838
2,2026-09-15,246.19,251.8150,246.1500,248.37,4518283
3,2026-09-14,249.41,251.3700,244.1201,249.09,6817275
4,2026-09-11,235.78,243.6300,235.0000,243.29,4919572
...,...,...,...,...,...,...
95,2026-05-01,234.55,235.9500,231.7500,232.20,3582091
96,2026-04-30,226.53,231.6000,224.9000,230.98,6283334
97,2026-04-29,230.34,231.4800,226.8200,227.10,6443724
98,2026-04-28,230.50,233.5550,228.4600,233.04,5162895


#**Paso 3.**

Manipulamos los datos para parsear correctamente la variable temporal y mostramos las estadísticas descriptivas de las variables que integran el DataFrame

In [5]:
Date = []

for index, row in df.iterrows():
  dia = datetime.strptime(row['timestamp'], '%Y-%m-%d')
  Date.append(dia)
df['date'] = pd.DataFrame(Date)
df.info()

dfw = pd.DataFrame()
dfw = df.drop(columns=['timestamp'])
dfw = dfw.reindex(columns=['date', 'open', 'high', 'low', 'close', 'volume'])
dfw.info()
dfw = dfw.iloc[::-1]
dfw.reset_index(drop=True, inplace=True)


last_date = dfw['date'].values[len(dfw)-1]
next_date = last_date + 86400000000000 # Este número es un día llevado a nanosegundos. Si se elige otro espaciado temporal entre las observaciones habría que ajustarlo

new_row = pd.DataFrame({ 'date': [next_date]})
dfw = pd.concat([dfw, new_row])
dfw.reset_index(drop=True, inplace=True)
dfw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  100 non-null    str           
 1   open       100 non-null    float64       
 2   high       100 non-null    float64       
 3   low        100 non-null    float64       
 4   close      100 non-null    float64       
 5   volume     100 non-null    int64         
 6   date       100 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 5.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    100 non-null    datetime64[us]
 1   open    100 non-null    float64       
 2   high    100 non-null    float64       
 3   low     100 non-null    float64       
 4   close   100 non-null    float64       
 5   vol

C:\Users\jorge.crespo.UNEATPDIPAS\AppData\Local\Temp\ipykernel_8404\3039918909.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  next_date = last_date + 86400000000000 # Este número es un día llevado a nanosegundos. Si se elige otro espaciado temporal entre las observaciones habría que ajustarlo


,date,open,high,low,close,volume
count,101,100.00000,100.000000,100.000000,100.000000,1.000000e+02
mean,2026-07-17 23:02:58.217821,245.61153,250.205765,241.631762,246.351400,9.032804e+06
min,2026-04-27 00:00:00,202.64000,207.550000,199.190000,205.770000,3.023636e+06
25%,2026-06-02 00:00:00,228.57750,230.992500,224.860000,227.925000,5.128232e+06
50%,2026-07-09 00:00:00,234.46500,238.667500,232.005000,235.635000,6.570695e+06
75%,2026-08-13 00:00:00,263.12500,268.875000,257.320000,264.400000,9.352911e+06
max,2029-06-13 00:00:00,322.55000,332.460000,310.109000,329.230000,6.744087e+07
std,NaN,27.88575,29.125129,26.480452,28.067403,8.134813e+06


In [6]:
dfw

,date,open,high,low,close,volume
0,2026-04-27,229.95,231.3200,227.1200,228.05,8060665.0
1,2026-04-28,230.50,233.5550,228.4600,233.04,5162895.0
2,2026-04-29,230.34,231.4800,226.8200,227.10,6443724.0
3,2026-04-30,226.53,231.6000,224.9000,230.98,6283334.0
4,2026-05-01,234.55,235.9500,231.7500,232.20,3582091.0
...,...,...,...,...,...,...
96,2026-09-14,249.41,251.3700,244.1201,249.09,6817275.0
97,2026-09-15,246.19,251.8150,246.1500,248.37,4518283.0
98,2026-09-16,240.50,243.3399,235.4100,237.49,6181838.0
99,2026-09-17,238.58,243.3650,237.3500,237.75,5905870.0


#**Paso 4.**

Dibujamos los gráficos de series temporales para el símbolo seleccionado (IBM)

In [7]:
fig = go.Figure(data=[go.Ohlc(x=df['date'],
                open=df['open'],
                high=df['high'],
                low=df['low'],
                close=df['close'])])
fig.update_layout(
    title='Gráfico Open, High, Low, Close',
    yaxis_title='IBM',
    xaxis_title='Fecha')

fig1 = px.line(df, x="date", y="close")
fig1.update_layout(
    title='Precio de Cierre',
    yaxis_title='IBM',
    xaxis_title='Fecha')

fig.show()
fig1.show()

#**Paso 5.**

Utilizaremos la librería [ta](https://technical-analysis-library-in-python.readthedocs.io/en/latest/) para calcular los indicadores técnicos para poder realizar análisis técnico del precio de la acción de IBM.

In [8]:
dfw['OPEN-1'] = dfw["open"].shift(1)
dfw['HIGH-1'] = dfw["high"].shift(1)
dfw['LOW-1'] = dfw["low"].shift(1)
dfw['CLOSE-1'] = dfw["close"].shift(1)
dfw['VOLUME-1'] = dfw["volume"].shift(1)

np.seterr(divide='ignore', invalid='ignore')

indicator_adx = ADXIndicator(close=dfw["CLOSE-1"], high=dfw["HIGH-1"], low=dfw["LOW-1"], window=14, fillna=True)
dfw['ADX14'] = indicator_adx.adx()

indicator_ema = EMAIndicator(close=dfw["CLOSE-1"], window=14)
dfw['EMA14'] = indicator_ema.ema_indicator()

indicator_sma7 = SMAIndicator(close=dfw["CLOSE-1"], window=7)
dfw['SMA7'] = indicator_sma7.sma_indicator()

indicator_sma26 = SMAIndicator(close=dfw["CLOSE-1"], window=26)
dfw['SMA26'] = indicator_sma26.sma_indicator()

indicator_sma42 = SMAIndicator(close=dfw["CLOSE-1"], window=42)
dfw['SMA42'] = indicator_sma42.sma_indicator()

indicator_macd = MACD(close=dfw["CLOSE-1"], window_slow=26, window_fast= 12, window_sign=9)
dfw['MACD'] = indicator_macd.macd()
dfw['MACD_SIGNAL'] =indicator_macd.macd_signal()

indicator_rsi = RSIIndicator(close=dfw["CLOSE-1"], window=14)
dfw['RSI'] =indicator_rsi.rsi()

indicator_cci = CCIIndicator(high=dfw["HIGH-1"], low=dfw["LOW-1"], close=dfw["CLOSE-1"], window=20, constant=0.15)
dfw['CCI'] =indicator_cci.cci()

indicator_RSIstochastic = StochRSIIndicator(close=dfw["CLOSE-1"], window=15, smooth1=3, smooth2=3)
dfw['RSIK'] =indicator_RSIstochastic.stochrsi_k()
dfw['RSID'] =indicator_RSIstochastic.stochrsi_d()

indicator_stochastic = StochasticOscillator(close=dfw["CLOSE-1"], high=dfw["HIGH-1"], low=dfw["LOW-1"], window=15, smooth_window=3)
dfw['K'] =indicator_stochastic.stoch()
dfw['D'] =indicator_stochastic.stoch_signal()
dfw['DIFF_K-D'] =indicator_stochastic.stoch()-indicator_stochastic.stoch_signal()

indicator_obv = OnBalanceVolumeIndicator(close=dfw["CLOSE-1"], volume=dfw["VOLUME-1"])
dfw['OBV'] =indicator_obv.on_balance_volume()

dfw['CLOSEDIFF'] =(dfw["CLOSE-1"]-dfw["OPEN-1"])/dfw["OPEN-1"]

ibm_TI_df = dfw
ibm_TI_df.info()

ibm_TI_df_topredict = ibm_TI_df[-1:]
ibm_TI_df = ibm_TI_df[:-1]
ibm_TI_df = ibm_TI_df.dropna()
ibm_TI_df.reset_index(drop=True, inplace=True)
ibm_TI_df.info()
ibm_TI_df

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         101 non-null    datetime64[us]
 1   open         100 non-null    float64       
 2   high         100 non-null    float64       
 3   low          100 non-null    float64       
 4   close        100 non-null    float64       
 5   volume       100 non-null    float64       
 6   OPEN-1       100 non-null    float64       
 7   HIGH-1       100 non-null    float64       
 8   LOW-1        100 non-null    float64       
 9   CLOSE-1      100 non-null    float64       
 10  VOLUME-1     100 non-null    float64       
 11  ADX14        101 non-null    float64       
 12  EMA14        87 non-null     float64       
 13  SMA7         94 non-null     float64       
 14  SMA26        75 non-null     float64       
 15  SMA42        59 non-null     float64       
 16  MACD         75 non

,date,open,high,low,close,volume,OPEN-1,HIGH-1,LOW-1,CLOSE-1,...,MACD_SIGNAL,RSI,CCI,RSIK,RSID,K,D,DIFF_K-D,OBV,CLOSEDIFF
0,2026-06-26,258.940,273.1350,258.2800,271.63,9719927.0,267.670,268.7559,256.0000,258.27,...,5.993491,47.318493,-6.818980,0.201035,0.141947,21.854404,23.880182,-2.025779,61212857.0,-0.035118
1,2026-06-29,274.300,278.1500,269.0600,278.00,6464204.0,258.940,273.1350,258.2800,271.63,...,5.188542,54.223766,-4.330194,0.353050,0.235795,47.679973,31.791228,15.888744,70932784.0,0.049007
2,2026-06-30,273.210,282.5694,271.1200,281.21,7458360.0,274.300,278.1500,269.0600,278.00,...,4.707584,57.110404,-1.154521,0.565028,0.373038,73.302008,47.612128,25.689880,77396988.0,0.013489
3,2026-07-01,279.660,294.4900,278.9600,286.25,6905811.0,273.210,282.5694,271.1200,281.21,...,4.495395,58.529594,1.216458,0.838169,0.585416,94.036582,71.672854,22.363728,84855348.0,0.029282
4,2026-07-02,283.140,290.9300,282.2800,289.52,5950158.0,279.660,294.4900,278.9600,286.25,...,4.532953,60.726918,7.475903,0.960440,0.787879,83.782720,83.707103,0.075617,91761159.0,0.023564
5,2026-07-06,288.345,300.8199,287.6500,299.52,7181525.0,283.140,290.9300,282.2800,289.52,...,4.766918,62.128972,9.712669,1.000000,0.932869,90.218461,89.345921,0.872540,97711317.0,0.022533
6,2026-07-07,305.660,311.8000,300.4901,306.13,8618569.0,288.345,300.8199,287.6500,299.52,...,5.260402,66.113142,16.195531,1.000000,0.986813,97.725057,90.575413,7.149645,104892842.0,0.038756
7,2026-07-08,300.770,303.8200,295.5900,302.05,7416749.0,305.660,311.8000,300.4901,306.13,...,5.984146,68.474081,20.754856,1.000000,1.000000,91.676453,93.206657,-1.530204,113511411.0,0.001538
8,2026-07-09,285.835,297.2838,284.4381,295.30,10841979.0,300.770,303.8200,295.5900,302.05,...,6.735541,65.443250,15.293121,0.962274,0.987425,85.687023,91.696178,-6.009155,106094662.0,0.004256
9,2026-07-10,297.260,298.7700,287.5000,287.56,3640089.0,285.835,297.2838,284.4381,295.30,...,7.341834,60.659560,9.433843,0.864833,0.942369,75.778039,84.380505,-8.602466,95252683.0,0.033114


#**Paso 6.**

Creamos un gráfico con algunos de los indicadores técnicos calculados.

In [9]:
fig2 = make_subplots(rows=3, cols=1)

fig2.append_trace(go.Candlestick(x=ibm_TI_df['date'],
                open=ibm_TI_df['open'],
                high=ibm_TI_df['high'],
                low=ibm_TI_df['low'],
                close=ibm_TI_df['close'], name='Gráfico de Velas'), row=1, col=1)
fig2.append_trace(go.Scatter(x=ibm_TI_df['date'], y=ibm_TI_df['close'], mode='lines', name='Cierre', line=dict(color='firebrick', width=3, dash='dot')), row=1, col=1)
fig2.append_trace(go.Scatter(x=ibm_TI_df['date'], y=ibm_TI_df['SMA7'], mode='lines', name='SMA7', line=dict(color='royalblue', width=3, dash='dot')), row=1, col=1)

fig2.append_trace(go.Scatter(x=ibm_TI_df['date'], y=ibm_TI_df['MACD'], mode='lines', name='MACD'), row=2, col=1)
fig2.append_trace(go.Scatter(x=ibm_TI_df['date'], y=ibm_TI_df['MACD_SIGNAL'], mode='lines', name='Señal MACD'), row=2, col=1)
fig2.add_hline(y=0, row=2, col=1, line=dict(color='gray', dash='dot'))

fig2.append_trace(go.Scatter(x=ibm_TI_df['date'], y=ibm_TI_df['RSI'], mode='lines', name='RSI'), row=3, col=1)
fig2.add_hline(y=80, row=3, col=1, line=dict(color='red', dash='dash'))
fig2.add_hline(y=20, row=3, col=1, line=dict(color='blue', dash='dash'))
fig2.update_layout(yaxis3 = dict(range=[0, 100]))

fig2.update_layout(
    height=800, width=1400,
    yaxis1_title='Velas y Media Móvil',
    yaxis2_title='MACD y Señal MACD',
    yaxis3_title='RSI',
    xaxis3_title='Fecha',
    xaxis1_rangeslider_visible=False)

fig2.show()

#**Paso 7.**

Ahora intentaremos hacer una predicción del precio futuro (un día por delante) del cierre de IBM. Utilizaremos dos modelos básicos:
* Regresión lineal múltiple sobre datos defasados
* Autoregresión (AutoARIMA)

In [10]:
TRAIN = ibm_TI_df.iloc[0:int(len(ibm_TI_df)//(1/0.90))]
TEST = ibm_TI_df.iloc[int(len(ibm_TI_df)//(1/0.90))+1:-1]

x_train =TRAIN.drop(columns=['open', 'close', 'high', 'low', 'volume', 'date'])
y_train = TRAIN['close']
x_test =TEST.drop(columns=['open', 'close', 'high', 'low', 'volume', 'date'])
y_test = TEST['close']

reg = LinearRegression().fit(x_train, y_train)
y_hat = reg.predict(x_test)
R2 = r2_score(y_test, y_hat)
if R2 < 0:
  print("El Modelo no es capaz de ajustar bien los datos, R^2 =",R2, "Una predicción arbitraria sería mejor")

x_predict = ibm_TI_df_topredict.drop(columns=['open', 'close', 'high', 'low', 'volume', 'date'])
prediction = reg.predict(x_predict).astype(float)

print("La predicción para el día ", next_date, "es ", prediction)

El Modelo no es capaz de ajustar bien los datos, R^2 = -2.565534252896955 Una predicción arbitraria sería mejor
La predicción para el día  2029-06-13T00:00:00.000000 es  [241.55172768]


In [11]:
X_train =ibm_TI_df['close']

model = auto_arima(X_train, start_p=1, start_q=1,
                      test='adf',
                      max_p=5, max_q=5,
                      m=1,
                      d=1,
                      seasonal=True,
                      start_P=0,
                      D=None,
                      trace=True,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)

Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=442.024, Time=0.08 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=438.169, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=440.136, Time=0.01 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=440.136, Time=0.02 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=436.338, Time=0.01 sec

Best model:  ARIMA(0,1,0)(0,0,0)[0]          
Total fit time: 0.131 seconds


In [12]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                   58
Model:               SARIMAX(0, 1, 0)   Log Likelihood                -217.169
Date:                Fri, 18 Sep 2026   AIC                            436.338
Time:                        09:51:14   BIC                            438.381
Sample:                             0   HQIC                           437.132
                                 - 58                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2       119.3540      5.391     22.138      0.000     108.787     129.921
===================================================================================
Ljung-Box (L1) (Q):                   0.04   Jarque-Bera (JB):              2598.45
Prob(Q):                              0.85   Prob(JB):                         0.00
Heteroskedasticity (H):               0.09   Skew:                            -5.08
Prob(H) (two-sided):                  0.00   Kurtosis:                        34.48
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [13]:
prediction, confint = model.predict(n_periods=1, return_conf_int=True)

print(prediction)
print(confint)

58    237.75
dtype: float64
[[216.33754339 259.16245661]]
